# EVOLVE – Annotation Reliability Analysis

This notebook implements the full inter-annotator reliability framework:

1. Initial Calibration (30 images)
2. Distributed Quality Assurance (70 images, block-based monitoring)
3. Final Random Audit (20 images)

Agreement is evaluated using:

- Count agreement
- Presence agreement (Cohen’s Kappa)
- Spatial agreement (Mean IoU)
- Temporal drift monitoring

## Setup

In [ ]:
# ==============================
# EVOLVE – Reliability Analysis
# ==============================

import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score
from IPython.display import display

# Resolve project root dynamically
ROOT = Path.cwd()
while not (ROOT / "data").exists():
    ROOT = ROOT.parent

print("Project root:", ROOT)

CLASS_NAMES = {
    0: "amp",
    1: "guitar",
    2: "drums",
    3: "micro",
    4: "mosh_pit",
    5: "hands_raised"
}

### Utility Functions

In [ ]:
# ------------------------------
# Label counting
# ------------------------------

def count_labels(folder_path):
    results = []
    files = sorted(os.listdir(folder_path))

    for file in files:
        image_name = file.replace(".txt", "")
        file_path = os.path.join(folder_path, file)

        counts = {class_name: 0 for class_name in CLASS_NAMES.values()}

        with open(file_path, "r") as f:
            lines = f.readlines()

            for line in lines:
                if line.strip() == "":
                    continue
                class_id = int(line.split()[0])
                class_name = CLASS_NAMES[class_id]
                counts[class_name] += 1

        row = {"image": image_name}
        row.update(counts)
        results.append(row)

    return pd.DataFrame(results)


# ------------------------------
# Count agreement
# ------------------------------

def compute_count_agreement(path_rina, path_anne):
    df_rina = count_labels(path_rina)
    df_anne = count_labels(path_anne)

    df = df_rina.merge(df_anne, on="image", suffixes=("_rina", "_anne"))

    summary = []

    for class_name in CLASS_NAMES.values():
        diff = df[f"{class_name}_rina"] - df[f"{class_name}_anne"]
        abs_diff = diff.abs()
        diff_rate = abs_diff / df[[f"{class_name}_rina", f"{class_name}_anne"]].max(axis=1).replace(0, 1)

        summary.append({
            "class": class_name,
            "mean_abs_diff": abs_diff.mean(),
            "mean_diff_rate": diff_rate.mean()
        })

    return pd.DataFrame(summary)


# ------------------------------
# Presence agreement (Kappa)
# ------------------------------

def compute_presence_kappa(path_rina, path_anne):
    df_rina = count_labels(path_rina)
    df_anne = count_labels(path_anne)

    df = df_rina.merge(df_anne, on="image", suffixes=("_rina", "_anne"))

    kappa_scores = {}

    for class_name in CLASS_NAMES.values():
        presence_rina = (df[f"{class_name}_rina"] > 0).astype(int)
        presence_anne = (df[f"{class_name}_anne"] > 0).astype(int)

        kappa_scores[class_name] = cohen_kappa_score(presence_rina, presence_anne)

    return pd.DataFrame.from_dict(kappa_scores, orient="index", columns=["kappa"])

### Spatial Agreement (IoU)

In [ ]:
# ------------------------------
# YOLO → bbox conversion
# ------------------------------

def yolo_to_bbox(row):
    _, x, y, w, h = row
    x1 = x - w / 2
    y1 = y - h / 2
    x2 = x + w / 2
    y2 = y + h / 2
    return [x1, y1, x2, y2]


def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_area = max(0, xB - xA) * max(0, yB - yA)

    boxA_area = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxB_area = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    union_area = boxA_area + boxB_area - inter_area

    if union_area == 0:
        return 0

    return inter_area / union_area


# ------------------------------
# Spatial agreement per class
# ------------------------------

def compute_spatial_agreement(path_rina, path_anne):

    files = sorted(os.listdir(path_rina))
    class_ious = {name: [] for name in CLASS_NAMES.values()}

    for file in files:

        rina_boxes = {}
        anne_boxes = {}

        for annotator, path in [("rina", path_rina), ("anne", path_anne)]:
            file_path = os.path.join(path, file)
            if not os.path.exists(file_path):
                continue

            with open(file_path, "r") as f:
                lines = f.readlines()

            boxes_per_class = {}

            for line in lines:
                parts = list(map(float, line.strip().split()))
                class_id = int(parts[0])
                class_name = CLASS_NAMES[class_id]
                bbox = yolo_to_bbox(parts)

                boxes_per_class.setdefault(class_name, []).append(bbox)

            if annotator == "rina":
                rina_boxes = boxes_per_class
            else:
                anne_boxes = boxes_per_class

        for class_name in CLASS_NAMES.values():

            r_boxes = rina_boxes.get(class_name, [])
            a_boxes = anne_boxes.get(class_name, [])

            used = set()

            for r_box in r_boxes:
                best_iou = 0
                best_idx = -1

                for i, a_box in enumerate(a_boxes):
                    if i in used:
                        continue
                    iou = compute_iou(r_box, a_box)
                    if iou > best_iou:
                        best_iou = iou
                        best_idx = i

                if best_idx >= 0:
                    used.add(best_idx)
                    class_ious[class_name].append(best_iou)

    mean_ious = {
        class_name: np.mean(vals) if len(vals) > 0 else np.nan
        for class_name, vals in class_ious.items()
    }

    return pd.DataFrame.from_dict(mean_ious, orient="index", columns=["mean_IoU"])

## 1. Calibration (30 Images)

Calibration was conducted on a stratified subset of 30 images
(10 very_low, 10 low, 10 medium luminance).

Objectives:
- Reduce subjectivity
- Harmonize bounding box definitions
- Detect systematic annotation bias

In [ ]:
CALIB_RINA = ROOT / "data/processed/calibration_30/labels_rina"
CALIB_ANNE = ROOT / "data/processed/calibration_30/labels_anne"

summary_calib = compute_count_agreement(CALIB_RINA, CALIB_ANNE)
kappa_calib = compute_presence_kappa(CALIB_RINA, CALIB_ANNE)
iou_calib = compute_spatial_agreement(CALIB_RINA, CALIB_ANNE)

print("=== Calibration – Count Agreement ===")
display(summary_calib)

print("=== Calibration – Presence Agreement (Kappa) ===")
display(kappa_calib)

print("=== Calibration – Spatial Agreement (IoU) ===")
display(iou_calib)

## 2. QA Block Monitoring

In [ ]:
QA_RINA = ROOT / "data/processed/qa_70/labels_rina"
QA_ANNE = ROOT / "data/processed/qa_70/labels_anne"

summary_qa = compute_count_agreement(QA_RINA, QA_ANNE)
kappa_qa = compute_presence_kappa(QA_RINA, QA_ANNE)
iou_qa = compute_spatial_agreement(QA_RINA, QA_ANNE)

display(summary_qa)
display(kappa_qa)
display(iou_qa)

### Drift Visualization

In [ ]:
plt.figure()
plt.bar(summary_qa["class"], summary_qa["mean_diff_rate"])
plt.xticks(rotation=45)
plt.title("QA – Mean Difference Rate")
plt.ylabel("Mean Difference Rate")
plt.show()

## 3. Final Audit

In [ ]:
AUDIT_RINA = ROOT / "data/processed/audit_20/labels_rina"
AUDIT_ANNE = ROOT / "data/processed/audit_20/labels_anne"

summary_audit = compute_count_agreement(AUDIT_RINA, AUDIT_ANNE)
kappa_audit = compute_presence_kappa(AUDIT_RINA, AUDIT_ANNE)
iou_audit = compute_spatial_agreement(AUDIT_RINA, AUDIT_ANNE)

display(summary_audit)
display(kappa_audit)
display(iou_audit)

### Final Summary Table

In [ ]:
final_summary = pd.concat({
    "Calibration": summary_calib.set_index("class")["mean_diff_rate"],
    "QA": summary_qa.set_index("class")["mean_diff_rate"],
    "Audit": summary_audit.set_index("class")["mean_diff_rate"]
}, axis=1)

final_summary